## STEP 1 — Load raw data ✅

In [4]:
import pandas as pd
import numpy as np

# Load the raw corrupted Posts dataset
posts = pd.read_csv('/Social_Engine_Posts_Corrupted.csv')

posts.shape

(12360, 8)

##STEP 2 — Create raw-data audit✅
Before cleaning the dataset, we first inspect its structure, columns, data types, and sample records.

This helps us understand the raw dataset and identify potential data-quality issues before making any changes.

In [27]:
# Display the first 5 rows of the raw dataset
# This gives us an initial look at the structure and values.
posts.head()

,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
0,to64mgey2v3y,user_vfxs1pry,Reddit,Bummed out with my new Air Max from Nike! Abso...,25-09-2024,4488.0,1456,673
1,7f0wdauzbj89,user_8l7rv5oe,Reddit,My one month review of Pepsi Crystal Pepsi: Hi...,1722528840,789.0,1484,39
2,dvvhg8eel45x,user_nfo3ih5u,NaN,Just unboxed my new Highlander from Toyota. Ex...,2025-04-13T20:12:18,NaN,1410,839
3,hgb9cxke4t7b,user_27aje6ur,Facebook,Comparing Pepsi Crystal Pepsi to the competiti...,10-09-2024,167.0,584,833
4,zwerpw3wk320,user_9px5q0by,Reddit,My one week review of Coca-Cola Diet Coke: Bes...,31-05-2024,4749.0,152,630


In [6]:
# Display all column names in the dataset
# This confirms which fields are available for analysis and cleaning.
posts.columns.tolist()

['post_id',
 'user_id',
 'platform',
 'text_content',
 'timestamp',
 'likes',
 'shares',
 'comments']

In [7]:
# Display the number of rows and columns
# Expected output: 12,360 rows and 8 columns.
print("Number of rows:", posts.shape[0])
print("Number of columns:", posts.shape[1])

Number of rows: 12360
Number of columns: 8


In [8]:
# Display column data types and the number of non-null values.
# This helps identify columns that may contain missing values
# and columns whose data type may need to be corrected later.
posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   post_id       12360 non-null  object 
 1   user_id       12360 non-null  object 
 2   platform      10514 non-null  object 
 3   text_content  10614 non-null  object 
 4   timestamp     12360 non-null  object 
 5   likes         10502 non-null  float64
 6   shares        12360 non-null  int64  
 7   comments      12360 non-null  int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 772.6+ KB


In [9]:
# Generate summary statistics for numerical columns.
# This helps us identify unusual values such as negative likes,
# unexpectedly large values, or unusual distributions.
posts.describe()

,likes,shares,comments
count,10502.000000,12360.000000,12360.000000
mean,2247.075795,1005.867557,504.081958
std,1797.342958,574.734542,288.796752
min,-4987.000000,0.000000,0.000000
25%,1038.250000,509.000000,252.000000
50%,2387.000000,1016.000000,503.000000
75%,3664.000000,1499.000000,755.000000
max,5000.000000,2000.000000,1000.000000


In [10]:
# Display the last 5 rows of the raw dataset.
# This helps confirm that the dataset was loaded completely
# and that the final records are not truncated.
posts.tail()

,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
12355,7nar7yotf6u4,user_zeija5lf,Facebook,NaN,2024-11-25T12:57:51,NaN,1663,870
12356,qokoc4of26wk,user_nd1s5fa0,Facebook,Just unboxed my new Highlander from Toyota. Di...,23-02-2025,NaN,1258,765
12357,jraym4uws51h,user_w66b5uk0,YouTube,NaN,2025-02-19T00:17:20,2515.0,1355,36
12358,21idyx41b5v4,user_tmjtxubu,Instagram,Just unboxed my new Vision Pro from Apple. Wor...,1742309918,2920.0,978,235
12359,2e9glp0jqtl1,user_u02jbwra,YouTube,My six months review of Coca-Cola Coke Zero: W...,2025-02-26T19:28:33,661.0,1052,297


### 2A. Duplicate Record Check

We check for duplicate rows and duplicate post IDs without modifying the raw dataset.

Duplicate records can lead to over-counting during analysis and may indicate data corruption.

In [11]:
# Check for completely identical rows
exact_duplicates = posts.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 360


In [12]:
# Check whether any post IDs occur more than once
duplicate_post_ids = posts['post_id'].duplicated().sum()

print("Rows with duplicate post IDs:", duplicate_post_ids)

Rows with duplicate post IDs: 360


In [13]:
# Display post IDs that occur more than once
duplicate_id_values = posts.loc[
    posts['post_id'].duplicated(keep=False),
    'post_id'
].value_counts()

duplicate_id_values.head(20)

,count
post_id,
4fgbp1fc2w0z,3
v3q8wgq6f29o,3
18872bs3czdy,3
3bj6zmt6mp0m,3
utkzs6e0k0pe,3
ajzrensvmuqq,3
am8fodsgv86c,3
vcx4pccxtll6,3
fs72byco9olr,2


### 2B. Categorical Value Check

We inspect categorical columns to identify their unique values and their frequencies.

This helps us detect unexpected categories, spelling variations, inconsistent capitalization, or other values that may indicate data corruption.

In [14]:
# Display the unique platform values and their frequencies.
# dropna=False ensures that missing platform values are also counted.
platform_values = posts['platform'].value_counts(dropna=False)

print("Platform values and their frequencies:")
print(platform_values)

Platform values and their frequencies:
platform
YouTube      2136
Facebook     2135
Twitter      2119
Reddit       2086
Instagram    2038
NaN          1846
Name: count, dtype: int64


In [15]:
# Count the number of distinct platform values, excluding missing values.
print("\nNumber of unique platforms:", posts['platform'].nunique())


Number of unique platforms: 5


### 2C. ID Format and Integrity Check

We inspect the post and user identifiers to ensure that they are present,
consistently formatted, and suitable for linking the Posts dataset with
the Users dataset.

No values are modified during this audit.

In [16]:
# Check for missing IDs
print("Missing post_id values:", posts['post_id'].isna().sum())
print("Missing user_id values:", posts['user_id'].isna().sum())

# Check the number of unique IDs
print("Unique post_id values:", posts['post_id'].nunique())
print("Unique user_id values:", posts['user_id'].nunique())

Missing post_id values: 0
Missing user_id values: 0
Unique post_id values: 12000
Unique user_id values: 1500


In [17]:
# Display a few sample IDs to check their structure
print("Sample post IDs:")
print(posts['post_id'].head(10).tolist())

print("\nSample user IDs:")
print(posts['user_id'].head(10).tolist())

Sample post IDs:
['to64mgey2v3y', '7f0wdauzbj89', 'dvvhg8eel45x', 'hgb9cxke4t7b', 'zwerpw3wk320', 'f85c8vm8ac8l', '19ys654yi1if', '5jywsz1bu0kp', 'ol0ai4w126p0', 'irjlew2pr8sw']

Sample user IDs:
['user_vfxs1pry', 'user_8l7rv5oe', 'user_nfo3ih5u', 'user_27aje6ur', 'user_9px5q0by', 'user_aedwfomw', 'user_27aje6ur', 'user_hvqqyzoi', 'user_634lriof', 'user_8v39sv2i']


In [18]:
# Check whether any post_id or user_id contains leading/trailing whitespace
post_id_whitespace = posts['post_id'].astype(str).str.strip().ne(posts['post_id'].astype(str)).sum()
user_id_whitespace = posts['user_id'].astype(str).str.strip().ne(posts['user_id'].astype(str)).sum()

print("post_id values with leading/trailing whitespace:", post_id_whitespace)
print("user_id values with leading/trailing whitespace:", user_id_whitespace)

post_id values with leading/trailing whitespace: 0
user_id values with leading/trailing whitespace: 0


###STEP 2D — Timestamp-format audit
The timestamp column may contain dates represented in different formats.

We inspect the raw values to identify the different timestamp formats,
invalid values, and the overall date range before applying any standardisation.

In [19]:
# Display a sample of the raw timestamp values
# This helps us visually identify the formats present in the dataset.
posts['timestamp'].head(20)

,timestamp
0,25-09-2024
1,1722528840
2,2025-04-13T20:12:18
3,10-09-2024
4,31-05-2024
5,24-01-2025
6,1719394663
7,2025-03-27T13:44:32
8,2024-05-05T05:52:34
9,11-02-2025


In [20]:
# Check how many timestamps appear to be numeric Unix timestamps.
# Unix timestamps are typically represented as numbers such as 1722528840.
numeric_timestamp = pd.to_numeric(posts['timestamp'], errors='coerce')

print("Numeric/Unix-style timestamps:", numeric_timestamp.notna().sum())
print("Non-numeric timestamp values:", numeric_timestamp.isna().sum())

Numeric/Unix-style timestamps: 3788
Non-numeric timestamp values: 8572


In [21]:
# Temporarily convert timestamps to datetime for auditing.
# The original timestamp column remains unchanged.
parsed_timestamp = pd.to_datetime(
    posts['timestamp'],
    errors='coerce',
    dayfirst=True
)

print("Earliest parsed timestamp:", parsed_timestamp.min())
print("Latest parsed timestamp:", parsed_timestamp.max())
print("Values that could not be parsed:", parsed_timestamp.isna().sum())

Earliest parsed timestamp: 2024-05-01 00:00:00
Latest parsed timestamp: 2025-04-30 00:00:00
Values that could not be parsed: 8738


In [22]:
# Classify timestamps based on their raw format.
# We are only auditing the data here; the original column is not modified.

timestamp_text = posts['timestamp'].astype(str).str.strip()

# Unix timestamps: digits only
unix_mask = timestamp_text.str.fullmatch(r'\d+')

# ISO-style timestamps: YYYY-MM-DD followed by time
iso_mask = timestamp_text.str.fullmatch(
    r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}'
)

# Date-only values in DD-MM-YYYY format
date_mask = timestamp_text.str.fullmatch(r'\d{2}-\d{2}-\d{4}')

# Anything that does not match the three known patterns
other_mask = ~(unix_mask | iso_mask | date_mask)

print("Unix-style timestamps:", unix_mask.sum())
print("ISO-style timestamps:", iso_mask.sum())
print("DD-MM-YYYY timestamps:", date_mask.sum())
print("Other/unrecognised formats:", other_mask.sum())

Unix-style timestamps: 3788
ISO-style timestamps: 4950
DD-MM-YYYY timestamps: 3622
Other/unrecognised formats: 0


######**Timestamp inconsistency:** The timestamp column contains three valid representations—Unix timestamp, ISO datetime, and DD-MM-YYYY. All records match one of these formats. This is a standardisation issue rather than missing or unparseable data.

### 2E. Text Content Sanity Check

We inspect the text content for missing or placeholder values, HTML artifacts,
encoding issues, and unusual whitespace.

These observations will guide the text-cleaning process in Step 7.
No text values are modified during this audit.

In [23]:
# Count missing values and empty/whitespace-only text entries
missing_text = posts['text_content'].isna().sum()

empty_text = (
    posts['text_content']
    .fillna('')
    .astype(str)
    .str.strip()
    .eq('')
    .sum()
)

print("Missing text values:", missing_text)
print("Empty/whitespace-only text values:", empty_text)

Missing text values: 1746
Empty/whitespace-only text values: 1746


In [24]:
# Display the most common text values.
# This helps identify placeholders such as NULL or N/A.
posts['text_content'].value_counts(dropna=False).head(20)

,count
text_content,
NaN,1746
NULL\n\n,24
NULL&amp;,21
NULLÃ©,19
NULL<div>,17
NULL<br>,13
"Just saw an ad for Apple Mac Mini during the CyberMonday. Wouldn't recommend. #Affordable, #Fitness, #Tech",3
Comparing Microsoft Xbox Elite Controller to the competition. Highly recommend. #Food Curious about your experience too.,3
"Loving it with my new Coca-Cola Cherry from Coca-Cola! It's okay. #MustHave, #Lifestyle, #SpecialOffer Curious about your experience too.<br>",3


In [25]:
# Check for common HTML tags or encoded HTML entities.
text = posts['text_content'].fillna('').astype(str)

html_tag_count = text.str.contains(r'<[^>]+>', regex=True).sum()
html_entity_count = text.str.contains(r'&[a-zA-Z]+;', regex=True).sum()

print("Rows containing HTML-like tags:", html_tag_count)
print("Rows containing HTML entities:", html_entity_count)

Rows containing HTML-like tags: 663
Rows containing HTML entities: 341


In [26]:
# Check for leading/trailing whitespace and embedded line breaks.
whitespace_count = (
    posts['text_content']
    .fillna('')
    .astype(str)
    .ne(posts['text_content'].fillna('').astype(str).str.strip())
    .sum()
)

linebreak_count = (
    posts['text_content']
    .fillna('')
    .astype(str)
    .str.contains(r'[\r\n]', regex=True)
    .sum()
)

print("Rows with leading/trailing whitespace:", whitespace_count)
print("Rows containing line breaks:", linebreak_count)

Rows with leading/trailing whitespace: 337
Rows containing line breaks: 337


##STEP 3 — Handle exact duplicates✅
Duplicate records can cause posts to be counted more than once during analysis.

We identify completely identical rows and remove only the redundant copies.
The original raw dataset is preserved, while cleaning is performed on a working copy.

In [28]:
# Create a separate working copy of the raw dataset.
# The original 'posts' DataFrame will remain unchanged.
posts_clean = posts.copy()

print("Rows before duplicate removal:", len(posts_clean))

Rows before duplicate removal: 12360


In [29]:
# Count rows that are exact duplicates of an earlier row.
# keep='first' means the first occurrence is treated as the original
# and subsequent identical rows are considered duplicates.
exact_duplicates = posts_clean.duplicated(keep='first').sum()

print("Exact duplicate rows found:", exact_duplicates)

Exact duplicate rows found: 360


In [30]:
# Remove exact duplicate rows while keeping the first occurrence.
posts_clean = posts_clean.drop_duplicates(keep='first').reset_index(drop=True)

print("Rows after duplicate removal:", len(posts_clean))
print("Rows removed:", len(posts) - len(posts_clean))

Rows after duplicate removal: 12000
Rows removed: 360


In [31]:
# Confirm that no exact duplicate rows remain.
remaining_duplicates = posts_clean.duplicated().sum()

print("Remaining exact duplicate rows:", remaining_duplicates)

Remaining exact duplicate rows: 0


## STEP 4 — Standardise Timestamps✅

The timestamp column contains three valid representations:
1. Unix timestamps
2. ISO datetime strings
3. DD-MM-YYYY date strings

We convert all three formats into a single datetime format while preserving
the original chronological information.

In [32]:
# Create a copy of the timestamp column for conversion.
# The raw timestamp column remains unchanged until we verify the conversion.
timestamp_raw = posts_clean['timestamp'].astype(str).str.strip()

# Identify Unix-style timestamps (digits only).
unix_mask = timestamp_raw.str.fullmatch(r'\d+')

# Create an empty datetime Series.
standard_timestamp = pd.Series(pd.NaT, index=posts_clean.index)

# Convert Unix timestamps to datetime.
standard_timestamp.loc[unix_mask] = pd.to_datetime(
    timestamp_raw.loc[unix_mask].astype('int64'),
    unit='s',
    errors='coerce'
)

# Convert ISO datetime values.
iso_mask = timestamp_raw.str.fullmatch(
    r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}'
)

standard_timestamp.loc[iso_mask] = pd.to_datetime(
    timestamp_raw.loc[iso_mask],
    errors='coerce'
)

# Convert DD-MM-YYYY values.
date_mask = timestamp_raw.str.fullmatch(r'\d{2}-\d{2}-\d{4}')

standard_timestamp.loc[date_mask] = pd.to_datetime(
    timestamp_raw.loc[date_mask],
    format='%d-%m-%Y',
    errors='coerce'
)

# Add the standardised timestamps to the working dataset.
posts_clean['timestamp'] = standard_timestamp

print("Unconverted timestamps:", posts_clean['timestamp'].isna().sum())

Unconverted timestamps: 0


In [33]:
# Verify the final timestamp range after standardisation.
print("Earliest timestamp:", posts_clean['timestamp'].min())
print("Latest timestamp:", posts_clean['timestamp'].max())

Earliest timestamp: 2024-05-01 00:00:00
Latest timestamp: 2025-04-30 21:57:10


In [34]:
# Confirm the timestamp column is now stored as a datetime type.
print("Timestamp data type:", posts_clean['timestamp'].dtype)

Timestamp data type: datetime64[ns]


## STEP 5 — Investigate Missing Values✅

We identify missing values in the cleaned working dataset and measure their
frequency and percentage.

The missing values are investigated before deciding whether they can be
recovered, replaced, or should remain missing.

In [35]:
# Count missing values in every column.
missing_count = posts_clean.isna().sum()

# Calculate the percentage of missing values in every column.
missing_percentage = (missing_count / len(posts_clean)) * 100

# Create a summary table for easier interpretation.
missing_summary = pd.DataFrame({
    'Missing Values': missing_count,
    'Missing Percentage': missing_percentage.round(2)
})

# Display only columns that contain missing values.
missing_summary[missing_summary['Missing Values'] > 0]

,Missing Values,Missing Percentage
platform,1784,14.87
text_content,1688,14.07
likes,1814,15.12


In [36]:
# Count how many rows have different numbers of missing fields.
# This helps us understand whether missing values are concentrated
# in the same records or spread across different records.

missing_per_row = posts_clean.isna().sum(axis=1)

print("Rows with no missing values:", (missing_per_row == 0).sum())
print("Rows with 1 missing value:", (missing_per_row == 1).sum())
print("Rows with 2 missing values:", (missing_per_row == 2).sum())
print("Rows with 3 missing values:", (missing_per_row == 3).sum())

Rows with no missing values: 7431
Rows with 1 missing value: 3891
Rows with 2 missing values: 639
Rows with 3 missing values: 39


In [37]:
# Display a sample of rows containing at least one missing value.
# We inspect the surrounding information before deciding how to handle them.

posts_clean[posts_clean.isna().any(axis=1)].head(20)

,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
2,dvvhg8eel45x,user_nfo3ih5u,NaN,Just unboxed my new Highlander from Toyota. Ex...,2025-04-13 20:12:18,NaN,1410,839
5,f85c8vm8ac8l,user_aedwfomw,NaN,Adidas ValentinesDeals is disappointing! Can't...,2025-01-24 00:00:00,3694.0,192,135
6,19ys654yi1if,user_27aje6ur,Facebook,Fed up with my new iPhone 15 from Apple! Best ...,2024-06-26 09:37:43,NaN,1578,514
10,cvdgyqy7xoq6,user_wv5x4t03,NaN,Just saw an ad for Samsung Galaxy Buds during ...,2024-06-11 14:54:47,2111.0,608,738
15,k9tvoyfmg8az,user_kf84zwv2,Facebook,NaN,2025-02-23 02:44:08,4153.0,450,319
16,cpiun4hvwfns,user_tb4fgpus,NaN,My two days review of Nike Dri-FIT: Not worth ...,2024-12-14 00:00:00,4815.0,360,312
20,mbykzpzh1l1y,user_750rlmnw,NaN,Could someone explain with my new Air Max from...,2024-07-16 02:21:20,-4812.0,1777,553
21,0eupqhh38ego,user_261d32en,Instagram,NaN,2025-03-19 04:05:04,2900.0,186,978
22,y2s75tpyz8xz,user_084pfdpp,YouTube,Just saw an ad for Pepsi Pepsi Max during the ...,2024-10-29 04:07:05,NaN,951,243
25,jhzjp26xujxh,user_lpaf67a1,YouTube,NaN,2024-05-11 08:08:20,3136.0,533,190


In [38]:
# Display a few complete examples from each column containing missing values.
# This helps us understand whether the missing data appears random or follows
# an identifiable pattern.

for column in ['platform', 'text_content', 'likes']:
    print(f"\n--- {column} ---")
    print(posts_clean[posts_clean[column].isna()].head(5))


--- platform ---
         post_id        user_id platform  \
2   dvvhg8eel45x  user_nfo3ih5u      NaN   
5   f85c8vm8ac8l  user_aedwfomw      NaN   
10  cvdgyqy7xoq6  user_wv5x4t03      NaN   
16  cpiun4hvwfns  user_tb4fgpus      NaN   
20  mbykzpzh1l1y  user_750rlmnw      NaN   

                                         text_content           timestamp  \
2   Just unboxed my new Highlander from Toyota. Ex... 2025-04-13 20:12:18   
5   Adidas ValentinesDeals is disappointing! Can't... 2025-01-24 00:00:00   
10  Just saw an ad for Samsung Galaxy Buds during ... 2024-06-11 14:54:47   
16  My two days review of Nike Dri-FIT: Not worth ... 2024-12-14 00:00:00   
20  Could someone explain with my new Air Max from... 2024-07-16 02:21:20   

     likes  shares  comments  
2      NaN    1410       839  
5   3694.0     192       135  
10  2111.0     608       738  
16  4815.0     360       312  
20 -4812.0    1777       553  

--- text_content ---
         post_id        user_id   platform tex

### 5A. Investigating Missing Platform Values

We examine posts with missing platform information to determine whether the
platform can be reliably recovered from other available fields.

Platform values will only be reconstructed when sufficient evidence exists.
Otherwise, the missing value will be retained rather than guessed.

In [39]:
# Select rows where the platform value is missing.
missing_platform = posts_clean[posts_clean['platform'].isna()]

# Display a sample of posts with missing platform information.
missing_platform[
    ['post_id', 'user_id', 'text_content', 'timestamp', 'likes', 'shares', 'comments']
].head(20)

,post_id,user_id,text_content,timestamp,likes,shares,comments
2,dvvhg8eel45x,user_nfo3ih5u,Just unboxed my new Highlander from Toyota. Ex...,2025-04-13 20:12:18,NaN,1410,839
5,f85c8vm8ac8l,user_aedwfomw,Adidas ValentinesDeals is disappointing! Can't...,2025-01-24 00:00:00,3694.0,192,135
10,cvdgyqy7xoq6,user_wv5x4t03,Just saw an ad for Samsung Galaxy Buds during ...,2024-06-11 14:54:47,2111.0,608,738
16,cpiun4hvwfns,user_tb4fgpus,My two days review of Nike Dri-FIT: Not worth ...,2024-12-14 00:00:00,4815.0,360,312
20,mbykzpzh1l1y,user_750rlmnw,Could someone explain with my new Air Max from...,2024-07-16 02:21:20,-4812.0,1777,553
35,i6io9jo0vpxa,user_y5urixy6,Comparing Pepsi Pepsi Lime to the competition....,2024-06-05 18:19:24,3407.0,649,991
39,d25e0a89eti4,user_va78mz39,Coca-Cola BlackFriday is okay! Can't wait to s...,2024-05-17 12:25:31,2980.0,955,382
42,sqdh2jbaeloe,user_ju8hqhh7,Comparing Adidas Samba to the competition. Exc...,2024-07-01 16:05:16,2011.0,925,30
44,juovcq3m427s,user_h39vyus8,NaN,2025-01-06 23:37:30,2127.0,1426,794
53,ym0fk6gqthz3,user_ivpoy0x1,Comparing Pepsi Pepsi Lime to the competition....,2025-02-02 17:07:23,3255.0,1498,481


In [40]:
# Look for explicit social-media platform names inside the text content.
# This is only an investigation; no values are changed.

platform_keywords = r'Facebook|Instagram|Twitter|YouTube|Reddit|LinkedIn|TikTok'

platform_mentions = missing_platform['text_content'].fillna('').str.contains(
    platform_keywords,
    case=False,
    regex=True
)

print("Missing-platform rows mentioning a platform:", platform_mentions.sum())
print("Total missing-platform rows:", len(missing_platform))

Missing-platform rows mentioning a platform: 0
Total missing-platform rows: 1784


### 5B. Investigating Missing Text Content

We examine records with missing text content to determine whether the
information can be recovered from other fields.

Text will not be fabricated or inferred from unrelated metadata.

In [41]:
# Select rows where text content is missing.
missing_text = posts_clean[posts_clean['text_content'].isna()]

# Display a sample of records with missing text.
missing_text[
    ['post_id', 'user_id', 'platform', 'timestamp', 'likes', 'shares', 'comments']
].head(20)

,post_id,user_id,platform,timestamp,likes,shares,comments
15,k9tvoyfmg8az,user_kf84zwv2,Facebook,2025-02-23 02:44:08,4153.0,450,319
21,0eupqhh38ego,user_261d32en,Instagram,2025-03-19 04:05:04,2900.0,186,978
25,jhzjp26xujxh,user_lpaf67a1,YouTube,2024-05-11 08:08:20,3136.0,533,190
33,bm0zjjwqdjdt,user_wolyc68e,Twitter,2024-06-15 07:07:50,2712.0,1150,436
36,aq3dtllhs6dr,user_2a5282v7,YouTube,2024-10-04 00:00:00,NaN,197,21
38,1fu318s17aoz,user_vdzbq60d,YouTube,2025-01-31 20:25:11,1690.0,1600,304
44,juovcq3m427s,user_h39vyus8,NaN,2025-01-06 23:37:30,2127.0,1426,794
57,6ix3i5q2grjy,user_a7y7wdlm,Facebook,2024-06-15 11:51:58,392.0,329,437
64,rcmwfuxl5rq7,user_4mf7je2u,Instagram,2024-06-21 00:25:10,2012.0,638,664
70,62q3q1y4rvye,user_ja1dmtpu,Facebook,2025-02-03 00:00:00,NaN,1594,365


In [42]:
# Check whether any missing-text post IDs have another record
# containing the same post ID.
text_recovery_check = posts_clean[
    posts_clean['post_id'].isin(missing_text['post_id'])
]

print("Total records matching missing-text post IDs:",
      len(text_recovery_check))

print("Unique missing-text post IDs:",
      missing_text['post_id'].nunique())

Total records matching missing-text post IDs: 1688
Unique missing-text post IDs: 1688


### 5C. Investigating Missing Likes

We examine records with missing likes to determine whether the missing
engagement value can be recovered from another occurrence of the same post.

Likes will not be estimated from shares, comments, or other unrelated fields.

In [43]:
# Select rows where likes are missing.
missing_likes = posts_clean[posts_clean['likes'].isna()]

print("Rows with missing likes:", len(missing_likes))
print("Unique post IDs with missing likes:", missing_likes['post_id'].nunique())

Rows with missing likes: 1814
Unique post IDs with missing likes: 1814


In [44]:
# Check whether any missing-like post IDs occur elsewhere in the dataset.
# If another record with the same post_id has a valid likes value,
# that value could potentially be used for recovery.

likes_recovery_check = posts_clean[
    posts_clean['post_id'].isin(missing_likes['post_id'])
]

print("Records matching missing-like post IDs:",
      len(likes_recovery_check))

Records matching missing-like post IDs: 1814


In [45]:
# Check whether any of the matching post IDs have a non-missing likes value.
recoverable_likes = likes_recovery_check[
    likes_recovery_check['likes'].notna()
]

print("Missing-like records with another valid likes value:",
      len(recoverable_likes))

Missing-like records with another valid likes value: 0


## STEP 6 — Fix Invalid Numerical Values✅

We inspect numerical engagement fields for values that violate logical
constraints.

Likes, shares, and comments represent counts of user interactions and
therefore cannot be negative.

Invalid values are identified first and will only be corrected after
their extent and pattern have been investigated.

### 6A. Identify Invalid Numerical Values

We examine the numerical engagement fields for values that violate
logical constraints.

Since likes, shares, and comments represent counts of interactions,
negative values are considered invalid. We first quantify these values
before applying any correction.

In [46]:
# Define the numerical engagement columns to audit.
numeric_columns = ['likes', 'shares', 'comments']

# Count negative values in each engagement column.
negative_summary = {}

for column in numeric_columns:
    negative_summary[column] = (posts_clean[column] < 0).sum()

print("Negative values by column:")
print(negative_summary)

Negative values by column:
{'likes': np.int64(509), 'shares': np.int64(0), 'comments': np.int64(0)}


In [47]:
# Display rows containing negative likes.
# We inspect the surrounding information before deciding how to correct them.

negative_likes = posts_clean[posts_clean['likes'] < 0]

negative_likes[
    ['post_id', 'user_id', 'platform', 'text_content',
     'timestamp', 'likes', 'shares', 'comments']
].head(20)

,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
20,mbykzpzh1l1y,user_750rlmnw,NaN,Could someone explain with my new Air Max from...,2024-07-16 02:21:20,-4812.0,1777,553
94,wal886ggp7kg,user_e3cwapff,YouTube,Loving it with my new Galaxy Tab from Samsung!...,2024-05-23 01:25:22,-1795.0,454,350
153,38tm1xfxxsdi,user_r7o639pp,Instagram,Should I upgrade about Samsung's Galaxy Buds? ...,2024-11-07 03:44:54,-3707.0,957,49
163,yktgh97ttawi,user_ral75hq0,Twitter,Has anyone else experienced battery problems w...,2024-10-27 19:52:17,-4543.0,363,703
188,loscndnlqi7b,user_xyvxp8vz,Twitter,Comparing Google Chromebook to the competition...,2025-04-11 07:26:02,-3194.0,904,378
192,hvjh643oxb8e,user_puypdmp1,YouTube,Just saw an ad for Apple iPad Air during the I...,2024-11-10 02:35:38,-3616.0,1728,363
197,0c0tc7wbhqdu,user_m2ziq5ox,Instagram,Comparing Google Nest Thermostat to the compet...,2025-03-14 22:02:13,-1165.0,709,403
263,dnhsqcx3t94w,user_oknxkzax,Twitter,Cannot believe with my new Galaxy Buds from Sa...,2025-02-04 18:06:24,-738.0,980,503
266,5fbt4nzjxol9,user_t2e8zk5w,Reddit,NaN,2024-09-04 22:36:26,-851.0,1959,185
268,whsvb3faauwf,user_uod57rwt,Facebook,Has anyone else experienced connectivity issue...,2025-02-07 14:38:01,-1205.0,1413,976


In [48]:
# Summarise the invalid likes values.
# This shows how extreme the negative values are.

print("Number of negative likes:", len(negative_likes))
print("Minimum likes value:", negative_likes['likes'].min())
print("Maximum negative likes value:", negative_likes['likes'].max())

Number of negative likes: 509
Minimum likes value: -4987.0
Maximum negative likes value: -11.0


In [49]:
# Check the minimum values of all numerical engagement fields.
# This confirms whether negative values occur only in likes
# or also appear in shares and comments.

for column in numeric_columns:
    print(f"{column}: minimum = {posts_clean[column].min()}")

likes: minimum = -4987.0
shares: minimum = 0
comments: minimum = 0


### 6B. Investigate Negative Likes for Possible Recovery

Negative values were found only in the likes column. Before replacing
these values, we check whether the affected posts appear elsewhere in
the dataset with a valid likes value.

If a valid value exists for the same post_id, it may provide a reliable
way to recover the corrupted value. If no such value exists, we will
treat the negative value as invalid rather than guessing its intended
value.

In [50]:
# Check whether any post IDs with negative likes appear elsewhere
# with a valid (non-negative) likes value.

negative_post_ids = negative_likes['post_id']

negative_like_recovery = posts_clean[
    posts_clean['post_id'].isin(negative_post_ids) &
    (posts_clean['likes'] >= 0)
]

print(
    "Negative-like records with another valid likes value:",
    len(negative_like_recovery)
)

Negative-like records with another valid likes value: 0


### 6C. Assess Relationship with Other Engagement Metrics

We examine the statistical relationship between likes, shares, and comments
to determine whether other engagement metrics provide evidence for recovering
invalid likes values.

This analysis is exploratory. Correlation is not used to fabricate or directly
replace individual likes values.

In [51]:
# Calculate the pairwise correlation between engagement metrics.
# This is used only to understand relationships within the dataset.

engagement_correlation = posts_clean[
    ['likes', 'shares', 'comments']
].corr()

engagement_correlation

,likes,shares,comments
likes,1.000000,0.010453,0.009679
shares,0.010453,1.000000,0.024406
comments,0.009679,0.024406,1.000000


### 6D. Correct Invalid Likes Values

The negative likes values cannot be reliably recovered from the available
data. Since negative engagement counts are invalid, these values are
converted to missing values (NaN).

This preserves data integrity by explicitly representing the true likes
count as unknown rather than introducing an estimated or fabricated value.

In [52]:
# Count negative likes before correction.
negative_before = (posts_clean['likes'] < 0).sum()

# Replace negative likes with NaN because their true values cannot
# be reliably recovered from the available dataset.
posts_clean.loc[posts_clean['likes'] < 0, 'likes'] = np.nan

# Verify the correction.
negative_after = (posts_clean['likes'] < 0).sum()

print("Negative likes before correction:", negative_before)
print("Negative likes after correction:", negative_after)
print("Missing likes after correction:", posts_clean['likes'].isna().sum())

Negative likes before correction: 509
Negative likes after correction: 0
Missing likes after correction: 2323


### 6E. Final Numerical Validation

We verify that all engagement values remaining in the cleaned dataset satisfy
the basic logical constraint that valid engagement counts cannot be negative.

Missing values are allowed because they represent unavailable information.

In [53]:
# Verify that no negative engagement values remain.
# Missing values (NaN) are excluded from the comparison.

for column in ['likes', 'shares', 'comments']:
    negative_count = (posts_clean[column] < 0).sum()
    print(f"{column} - negative values remaining: {negative_count}")

# Display the minimum valid value for each engagement metric.
print("\nMinimum valid values:")
print(posts_clean[['likes', 'shares', 'comments']].min())

likes - negative values remaining: 0
shares - negative values remaining: 0
comments - negative values remaining: 0

Minimum valid values:
likes       0.0
shares      0.0
comments    0.0
dtype: float64


## STEP 7 — Clean Text / HTML Artifacts✅

The text_content column is inspected for HTML tags, HTML entities,
leading/trailing whitespace, and line breaks.

These formatting artifacts are removed or standardised while preserving
the original textual meaning. Missing text values are left unchanged.

In [54]:
# Create a copy of the text column before cleaning.
# This allows the original cleaned-dataset state to be compared if needed.

text_before = posts_clean['text_content'].copy()

In [55]:
import re
import html

# Define a function to clean HTML-related artifacts from text.
def clean_text(text):
    # Leave missing text values unchanged.
    if pd.isna(text):
        return text

    # Convert the value to a string.
    text = str(text)

    # Remove HTML tags such as <br>, <p>, <div>, etc.
    text = re.sub(r'<[^>]+>', ' ', text)

    # Decode HTML entities such as &amp;, &lt;, &gt;, etc.
    text = html.unescape(text)

    # Replace line breaks and tabs with a single space.
    text = re.sub(r'[\r\n\t]+', ' ', text)

    # Remove unnecessary repeated whitespace.
    text = re.sub(r'\s+', ' ', text)

    # Remove leading and trailing whitespace.
    text = text.strip()

    return text


# Apply the cleaning function to the text_content column.
posts_clean['text_content'] = posts_clean['text_content'].apply(clean_text)

In [56]:
# Check whether HTML-like tags remain after cleaning.
remaining_html_tags = posts_clean['text_content'].fillna('').str.contains(
    r'<[^>]+>',
    regex=True
).sum()

# Check whether common HTML entities remain.
remaining_html_entities = posts_clean['text_content'].fillna('').str.contains(
    r'&(?:amp|lt|gt|quot|apos|nbsp);',
    regex=True,
    case=False
).sum()

# Check whether line breaks remain.
remaining_line_breaks = posts_clean['text_content'].fillna('').str.contains(
    r'[\r\n\t]',
    regex=True
).sum()

print("HTML-like tags remaining:", remaining_html_tags)
print("HTML entities remaining:", remaining_html_entities)
print("Line breaks remaining:", remaining_line_breaks)

HTML-like tags remaining: 0
HTML entities remaining: 0
Line breaks remaining: 0


In [57]:
# Check for leading or trailing whitespace in non-missing text values.
whitespace_issues = posts_clean['text_content'].dropna().apply(
    lambda x: x != x.strip()
).sum()

print("Rows with leading/trailing whitespace remaining:", whitespace_issues)

Rows with leading/trailing whitespace remaining: 0


## STEP 8 — Validate Against Users Dataset✅

The Posts dataset contains a user_id field that should correspond to a
valid user in the Users dataset.

We cross-check the user_id values between both datasets to identify posts
that reference users who are not present in the Users table.

In [59]:
# Load the Users dataset for cross-validation.
users = pd.read_csv('/Social_Engine_Users.csv')

# Display the dimensions of the Users dataset.
print("Users dataset shape:", users.shape)

# Display the column names.
print("\nUsers dataset columns:")
print(users.columns.tolist())

Users dataset shape: (1500, 5)

Users dataset columns:
['user_id', 'location', 'language', 'account_created', 'follower_count']


### Check Users Dataset User IDs

The Users dataset is used as the reference table for validating post
ownership. We first verify that its user_id values are complete and unique.

In [60]:
# Check for missing user IDs in the Users dataset.
print("Missing user_id values:", users['user_id'].isna().sum())

# Check for duplicate user IDs.
print("Duplicate user_id values:", users['user_id'].duplicated().sum())

# Count unique user IDs.
print("Unique user_id values:", users['user_id'].nunique())

# Count total user records.
print("Total user records:", len(users))

Missing user_id values: 0
Duplicate user_id values: 0
Unique user_id values: 1500
Total user records: 1500


In [61]:
# Create a set of valid user IDs from the Users dataset.
valid_user_ids = set(users['user_id'])

# Identify posts whose user_id does not exist in the Users dataset.
invalid_user_posts = posts_clean[
    ~posts_clean['user_id'].isin(valid_user_ids)
]

print("Posts with user_id not found in Users dataset:",
      len(invalid_user_posts))

print("Unique invalid user_ids:",
      invalid_user_posts['user_id'].nunique())

Posts with user_id not found in Users dataset: 0
Unique invalid user_ids: 0


### Check User–Post Coverage

We compare the Users and Posts datasets to determine how many registered
users have at least one post and how many users have no associated posts.

Users without posts are not automatically treated as invalid because an
inactive user may legitimately have no post records.

In [62]:
# Count the number of posts associated with each user.
posts_per_user = posts_clean.groupby('user_id').size()

# Identify users from the Users dataset who have no posts.
users_without_posts = users[
    ~users['user_id'].isin(posts_per_user.index)
]

# Display the coverage results.
print("Total users:", len(users))
print("Users with at least one post:", len(posts_per_user))
print("Users without posts:", len(users_without_posts))

Total users: 1500
Users with at least one post: 1500
Users without posts: 0


In [63]:
# Show the distribution of the number of posts per user.
# This helps identify whether posts are concentrated among a small number
# of users or distributed across the user population.

posts_per_user.describe()

,0
count,1500.000000
mean,8.000000
std,2.851214
min,1.000000
25%,6.000000
50%,8.000000
75%,10.000000
max,22.000000


## STEP 9 — Final Quality Checks✅

The cleaned Posts dataset is subjected to final consistency and integrity
checks before export.

The checks verify that duplicate records have been removed, numerical
constraints are satisfied, timestamps are valid, text artifacts have been
cleaned, missing values are documented, and all user references remain
valid.

In [64]:
# Check the final number of rows and columns.
print("Final dataset shape:", posts_clean.shape)

# Check for remaining exact duplicate rows.
print("Exact duplicate rows:", posts_clean.duplicated().sum())

# Check for duplicate post IDs.
print("Duplicate post IDs:", posts_clean['post_id'].duplicated().sum())

Final dataset shape: (12000, 8)
Exact duplicate rows: 0
Duplicate post IDs: 0


In [65]:
# Check the final missing-value counts after all cleaning operations.
# These values are reported rather than automatically filled because
# the original information could not be reliably recovered.

final_missing = posts_clean.isna().sum()

final_missing_summary = pd.DataFrame({
    'Missing Values': final_missing,
    'Missing Percentage': (
        final_missing / len(posts_clean) * 100
    ).round(2)
})

final_missing_summary[final_missing_summary['Missing Values'] > 0]

,Missing Values,Missing Percentage
platform,1784,14.87
text_content,1688,14.07
likes,2323,19.36


In [66]:
# Check that no negative engagement values remain.

for column in ['likes', 'shares', 'comments']:
    negative_count = (posts_clean[column] < 0).sum()
    print(f"{column} - negative values remaining: {negative_count}")

# Display the minimum valid value in each engagement column.
print("\nMinimum valid values:")
print(posts_clean[['likes', 'shares', 'comments']].min())

likes - negative values remaining: 0
shares - negative values remaining: 0
comments - negative values remaining: 0

Minimum valid values:
likes       0.0
shares      0.0
comments    0.0
dtype: float64


In [67]:
# Check the timestamp data type.
print("Timestamp data type:", posts_clean['timestamp'].dtype)

# Count any missing timestamps after standardisation.
print("Missing timestamps:", posts_clean['timestamp'].isna().sum())

# Check the final timestamp range.
print("Earliest timestamp:", posts_clean['timestamp'].min())
print("Latest timestamp:", posts_clean['timestamp'].max())

Timestamp data type: datetime64[ns]
Missing timestamps: 0
Earliest timestamp: 2024-05-01 00:00:00
Latest timestamp: 2025-04-30 21:57:10


In [68]:
# Check for remaining HTML-like tags.
remaining_html_tags = posts_clean['text_content'].fillna('').str.contains(
    r'<[^>]+>',
    regex=True
).sum()

# Check for remaining common HTML entities.
remaining_html_entities = posts_clean['text_content'].fillna('').str.contains(
    r'&(?:amp|lt|gt|quot|apos|nbsp);',
    regex=True,
    case=False
).sum()

# Check for remaining line breaks or tabs.
remaining_line_breaks = posts_clean['text_content'].fillna('').str.contains(
    r'[\r\n\t]',
    regex=True
).sum()

# Check that every user_id exists in the Users dataset.
invalid_user_ids = (~posts_clean['user_id'].isin(valid_user_ids)).sum()

print("HTML-like tags remaining:", remaining_html_tags)
print("HTML entities remaining:", remaining_html_entities)
print("Line breaks/tabs remaining:", remaining_line_breaks)
print("Invalid user_id values:", invalid_user_ids)

HTML-like tags remaining: 0
HTML entities remaining: 0
Line breaks/tabs remaining: 0
Invalid user_id values: 0


In [69]:
# Display the final column names and data types.
print("Final columns:")
print(posts_clean.columns.tolist())

print("\nFinal data types:")
print(posts_clean.dtypes)

# Verify that the expected 8 columns are present.
expected_columns = [
    'post_id',
    'user_id',
    'platform',
    'text_content',
    'timestamp',
    'likes',
    'shares',
    'comments'
]

missing_columns = set(expected_columns) - set(posts_clean.columns)
unexpected_columns = set(posts_clean.columns) - set(expected_columns)

print("\nMissing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

# Verify important data types.
print("\nData-type checks:")
print("post_id is object:", posts_clean['post_id'].dtype == 'object')
print("user_id is object:", posts_clean['user_id'].dtype == 'object')
print("timestamp is datetime:", pd.api.types.is_datetime64_any_dtype(posts_clean['timestamp']))
print("likes is numeric:", pd.api.types.is_numeric_dtype(posts_clean['likes']))
print("shares is numeric:", pd.api.types.is_numeric_dtype(posts_clean['shares']))
print("comments is numeric:", pd.api.types.is_numeric_dtype(posts_clean['comments']))

Final columns:
['post_id', 'user_id', 'platform', 'text_content', 'timestamp', 'likes', 'shares', 'comments']

Final data types:
post_id                 object
user_id                 object
platform                object
text_content            object
timestamp       datetime64[ns]
likes                  float64
shares                   int64
comments                 int64
dtype: object

Missing expected columns: set()
Unexpected columns: set()

Data-type checks:
post_id is object: True
user_id is object: True
timestamp is datetime: True
likes is numeric: True
shares is numeric: True
comments is numeric: True


## STEP 10 — Export Cleaned Dataset✅

The validated and cleaned Posts dataset is exported as a CSV file for
submission.

Missing values are retained as empty CSV fields because their original
values could not be reliably recovered. No data is fabricated during export.

In [70]:
# Export the final cleaned Posts dataset to CSV.
# NaN values are written as empty fields in the CSV.

output_file = 'Social_Engine_Posts_Cleaned.csv'

posts_clean.to_csv(
    output_file,
    index=False
)

print("Cleaned dataset exported successfully.")
print("File:", output_file)
print("Rows:", len(posts_clean))
print("Columns:", len(posts_clean.columns))

Cleaned dataset exported successfully.
File: Social_Engine_Posts_Cleaned.csv
Rows: 12000
Columns: 8


In [71]:
# Reload the exported CSV file.
posts_final = pd.read_csv('Social_Engine_Posts_Cleaned.csv')

# Check the saved dataset dimensions.
print("Reloaded dataset shape:", posts_final.shape)

# Check the column names.
print("\nReloaded columns:")
print(posts_final.columns.tolist())

# Check for exact duplicate rows after export.
print("\nExact duplicate rows:", posts_final.duplicated().sum())

# Check for duplicate post IDs.
print("Duplicate post IDs:", posts_final['post_id'].duplicated().sum())

# Check for negative engagement values after export.
print("\nNegative engagement values:")
for column in ['likes', 'shares', 'comments']:
    print(f"{column}:", (posts_final[column] < 0).sum())

# Check missing values after export.
print("\nMissing values:")
print(posts_final.isna().sum())

Reloaded dataset shape: (12000, 8)

Reloaded columns:
['post_id', 'user_id', 'platform', 'text_content', 'timestamp', 'likes', 'shares', 'comments']

Exact duplicate rows: 0
Duplicate post IDs: 0

Negative engagement values:
likes: 0
shares: 0
comments: 0

Missing values:
post_id            0
user_id            0
platform        1784
text_content    1740
timestamp          0
likes           2323
shares             0
comments           0
dtype: int64


In [72]:
from google.colab import files

files.download('Social_Engine_Posts_Cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>